In [1]:
import pandas as pd
print("pandas version:", pd.__version__)

pandas version: 2.3.3


In [2]:
import pandas as pd
import datasets
from datasets import load_dataset
import numpy as np

c:\Users\hp\Desktop\med\venv_med\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("bigbio/bc5cdr",trust_remote_code=True)

In [ ]:
mt_data=pd.read_csv(r'C:\Users\hp\Desktop\med\mtsamples.csv\mtsamples.csv')

In [ ]:
print(mt_data.head(1))


In [ ]:
print(dataset)


In [ ]:

print(dataset["train"][0])


In [ ]:

print(dataset["train"][0].keys())


In [ ]:

print(dataset["train"][499]["passages"][0]["type"])


In [ ]:

sample=dataset['train'][0]['passages']

for passage in sample:
    print(passage['type'])
    print(passage['text'])

In [ ]:
s=dataset['train']

for i in range(500):
    for p in dataset['train'][i]['passages']:
        if p['type']=='title':
            print(i+1,p['entities'][0]['offsets'])
            print(i+1,p['entities'][1]['offsets'])

In [ ]:
first=dataset['train'][0]

print(first.keys())
print(len(dataset['train']))

In [ ]:
first_passage=dataset['train'][0]['passages'][0]

print(first_passage['document_id'])
print(first_passage['type'])
print(first_passage['text'])
for e in first_passage['entities']:
    print(e , " ")
    
second_passage=dataset['train'][0]['passages'][1]

print(second_passage['document_id'])
print(second_passage['type'])
print(second_passage['text'])
for e in second_passage['entities']:
    print(e , " ")
    

In [ ]:
# Let's find a passage that has BOTH Chemical and Disease entities
for i, row in enumerate(dataset['train']):
    for p in row['passages']:
        types = [e['type'] for e in p['entities']]
        if 'Chemical' in types and 'Disease' in types:
            print(f"Found in row {i}:")
            print("Text:", p['text'])
            print()
            for e in p['entities']:
                print(f"  [{e['type']}] '{e['text']}' at offsets {e['offsets']}")
            break
    else:
        continue
    break

In [4]:
import transformers
import torch

In [5]:

print("datasets version:", datasets.__version__)
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("All packages loaded successfully!")

datasets version: 2.18.0
transformers version: 4.57.6
torch version: 2.12.0+cpu
All packages loaded successfully!


In [ ]:
total_passages = 0
total_entities = 0

for row in dataset['train']:
    for passage in row['passages']:
        total_passages += 1
        total_entities += len(passage['entities'])

print("Total passages in training set:", total_passages)
print("Total entities in training set:", total_entities)
print("Average entities per passage  :", round(total_entities / total_passages, 2))

In [ ]:
row = dataset['train'][0]

for passage in row['passages']:
    print("=== PASSAGE ===")
    print("Type   :", passage['type'])
    print("Text   :", passage['text'])
    print()
    print("Entities:")
    for entity in passage['entities']:
        print("  ", entity)
    print()

In [ ]:
total_passages = 0
total_entities = 0

for row in dataset['train']:
    for passage in row['passages']:
        total_passages += 1
        total_entities += len(passage['entities'])

print("Total passages in training set:", total_passages)
print("Total entities in training set:", total_entities)
print("Average entities per passage  :", round(total_entities / total_passages, 2))

In [6]:
from transformers import AutoTokenizer

# AutoTokenizer automatically downloads the right tokenizer
# for whatever model name we give it
# "dmis-lab/biobert-base-cased-v1.2" is BioBERT —
# a version of BERT that was trained on biomedical text
# "cased" means it treats "Naloxone" and "naloxone" as different words
# which matters in medical text where capitalization can carry meaning

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2",local_files_only=True)

print("Tokenizer loaded!")
print("Vocabulary size:", tokenizer.vocab_size)


Tokenizer loaded!
Vocabulary size: 28996


In [ ]:
# Let's see exactly how BioBERT splits our example sentence
# This is the core thing we need to understand before writing BIO conversion

sentence = "Naloxone reverses the antihypertensive effect of clonidine."

# tokenize() just splits into tokens — no numbers yet, just words
tokens = tokenizer.tokenize(sentence)
print("Tokens:", tokens)
print("Count :", len(tokens))
print()

# Now let's see a tricky medical word
word = "antihypertensive"
tokens2 = tokenizer.tokenize(word)
print(f"How '{word}' gets split:")
print(tokens2)
print()

# And a hyphenated chemical name
word2 = "alpha-methyldopa"
tokens3 = tokenizer.tokenize(word2)
print(f"How '{word2}' gets split:")
print(tokens3)

In [7]:
def preprocess_passage(text, entities, tokenizer):

    label2id = {
        "O":          0,
        "B-Chemical": 1,
        "I-Chemical": 2,
        "B-Disease":  3,
        "I-Disease":  4,
    }

    # One label per character, default O
    char_labels = ["O"] * len(text)

    for entity in entities:
        for start, end in entity["offsets"]:

            # ── FIX: skip if offsets go beyond text length ──
            # Some passages have annotation errors where the
            # offset points outside the actual text string
            # We simply skip those broken entities
            if start >= len(text) or end > len(text):
                continue

            entity_type = entity["type"]

            char_labels[start] = f"B-{entity_type}"

            for i in range(start + 1, end):
                char_labels[i] = f"I-{entity_type}"

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=True,
        max_length=512
    )

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offset_mapping = encoding["offset_mapping"]

    token_labels = []

    for token, (char_start, char_end) in zip(tokens, offset_mapping):

        if char_start == char_end:
            token_labels.append(-100)
            continue

        if token.startswith("##"):
            token_labels.append(-100)
            continue

        label_str = char_labels[char_start]
        token_labels.append(label2id[label_str])

    return tokens, token_labels


print("Fixed function defined successfully!")

Fixed function defined successfully!


In [ ]:
# Let's test on the title passage we've been studying
# "Naloxone reverses the antihypertensive effect of clonidine."

# Get the first passage from training row 0
passage = dataset['train'][0]['passages'][0]

text     = passage['text']
entities = passage['entities']

print("Text:", text)
print()
print("Entities:")
for e in entities:
    print(" ", e['text'], "→ type:", e['type'], "→ offsets:", e['offsets'])
print()

# Run our preprocessing function
tokens, labels = preprocess_passage(text, entities, tokenizer)

# id2label lets us convert numbers back to readable label names
id2label = {
    0:    "O",
     1:   "B-Chemical",
     2:   "I-Chemical",
     3:   "B-Disease",
     4:   "I-Disease",
    -100: "[IGNORE]"
}

# Print every token alongside its label so we can verify
print(f"{'TOKEN':<20} {'LABEL'}")
print("-" * 35)
for token, label in zip(tokens, labels):
    print(f"{token:<20} {id2label[label]}")


In [ ]:
# We'll store all processed passages in a list
# Each item will be a dict with tokens and labels for one passage

all_tokens = []   # list of token lists
all_labels = []   # list of label lists
skipped    = 0    # count passages we skip (empty ones)

# Loop through every row in the training set
for row in dataset['train']:

    # Each row has multiple passages (title + abstract)
    for passage in row['passages']:

        text     = passage['text']
        entities = passage['entities']

        # Skip if text is empty — occasionally happens
        if not text.strip():
            skipped += 1
            continue

        # Run our preprocessing function on this passage
        tokens, labels = preprocess_passage(text, entities, tokenizer)

        # Only keep passages that have at least one token
        if len(tokens) > 0:
            all_tokens.append(tokens)
            all_labels.append(labels)

print(f"Total passages processed : {len(all_tokens)}")
print(f"Passages skipped         : {skipped}")
print()

# Quick check — show stats about label distribution
from collections import Counter

# Flatten all labels into one big list to count them
flat_labels = [l for sublist in all_labels for l in sublist]

id2label = {
    0: "O",
    1: "B-Chemical",
    2: "I-Chemical",
    3: "B-Disease",
    4: "I-Disease",
    -100: "[IGNORE]"
}

print("Label distribution across all tokens:")
counts = Counter(flat_labels)
for label_id, count in sorted(counts.items()):
    label_name = id2label[label_id]
    print(f"  {label_name:<15} : {count:>8,}")

In [9]:
# import pickle
# import os

# # Create the folder if it doesn't exist
# save_dir = r"C:\Users\hp\Desktop\snomed_ner"
# os.makedirs(save_dir, exist_ok=True)
# # exist_ok=True means: if folder already exists, don't throw an error
# # if it doesn't exist, create it

# save_path = os.path.join(save_dir, "preprocessed_bc5cdr.pkl")

# # Save
# with open(save_path, "wb") as f:
#     pickle.dump({
#         "tokens": all_tokens,
#         "labels": all_labels
#     }, f)

# print(f"Saved {len(all_tokens)} passages to disk")
# print(f"Location: {save_path}")

# # Verify it loads back correctly
# with open(save_path, "rb") as f:
#     loaded = pickle.load(f)

# print(f"Verified — loaded back {len(loaded['tokens'])} passages")
# print()
# print("Sample — first passage tokens:", loaded['tokens'][0])
# print("Sample — first passage labels:", loaded['labels'][0])


import pickle
from datasets import load_dataset

dataset = load_dataset("bigbio/bc5cdr", trust_remote_code=True)

with open(r"C:\Users\hp\Desktop\snomed_ner\preprocessed_bc5cdr.pkl", "rb") as f:
    loaded = pickle.load(f)

all_tokens = loaded['tokens']
all_labels = loaded['labels']

print(f"Loaded {len(all_tokens)} passages from disk")

EOFError: Ran out of input

In [ ]:
# Right now all_tokens contains string tokens like ['na','##lo',...]
# The model doesn't understand strings — only numbers
# We need to convert every token string → its integer ID in BioBERT's vocabulary

# Also we need to add special tokens:
# [CLS] at the start — tells model "beginning of sequence"
# [SEP] at the end   — tells model "end of sequence"
# These are BioBERT's required format

all_input_ids = []   # token strings → integer IDs
all_label_ids = []   # labels aligned with the new tokens including CLS/SEP

CLS_ID = tokenizer.cls_token_id   # the integer ID for [CLS]
SEP_ID = tokenizer.sep_token_id   # the integer ID for [SEP]

print("CLS token:", tokenizer.cls_token, "→ ID:", CLS_ID)
print("SEP token:", tokenizer.sep_token, "→ ID:", SEP_ID)
print()

for tokens, labels in zip(all_tokens, all_labels):

    # Convert token strings to their integer IDs
    input_ids = tokenizer.convert_tokens_to_ids(tokens)

    # Add [CLS] at start and [SEP] at end
    # CLS and SEP are special tokens — label them -100 (ignore)
    input_ids = [CLS_ID] + input_ids + [SEP_ID]
    label_ids = [-100]   + labels    + [-100]

    all_input_ids.append(input_ids)
    all_label_ids.append(label_ids)

print(f"Total sequences ready : {len(all_input_ids)}")
print()
print("First sequence input_ids:", all_input_ids[0])
print("First sequence label_ids:", all_label_ids[0])
print()
print("Length of first input_ids:", len(all_input_ids[0]))
print("Length of first label_ids:", len(all_label_ids[0]))
print("Both lengths match:", len(all_input_ids[0]) == len(all_label_ids[0]))

In [ ]:
# Find the longest sequence in our dataset
max_len = max(len(ids) for ids in all_input_ids)
print("Longest sequence:", max_len, "tokens")
print("Shortest sequence:", min(len(ids) for ids in all_input_ids), "tokens")
print()

# PAD_ID is the token ID BioBERT uses for padding
# It's always 0 in BERT-style models
PAD_ID = tokenizer.pad_token_id
print("PAD token:", tokenizer.pad_token, "→ ID:", PAD_ID)
print()

all_input_ids_padded    = []  # padded input token IDs
all_attention_masks     = []  # 1 = real token, 0 = padding
all_label_ids_padded    = []  # padded labels

for input_ids, label_ids in zip(all_input_ids, all_label_ids):

    # How many padding tokens do we need to add?
    pad_length = max_len - len(input_ids)

    # Pad input_ids with PAD_ID (0)
    padded_input  = input_ids + [PAD_ID] * pad_length

    # Attention mask: 1 for real tokens, 0 for padding
    # The model uses this to ignore padding positions
    attention_mask = [1] * len(input_ids) + [0] * pad_length

    # Pad labels with -100 so padding positions are ignored in loss
    padded_labels = label_ids + [-100] * pad_length

    all_input_ids_padded.append(padded_input)
    all_attention_masks.append(attention_mask)
    all_label_ids_padded.append(padded_labels)

# Verify
print("First sequence after padding:")
print("  input_ids length    :", len(all_input_ids_padded[0]))
print("  attention_mask length:", len(all_attention_masks[0]))
print("  label_ids length    :", len(all_label_ids_padded[0]))
print()
print("All same length?", 
    len(set(len(x) for x in all_input_ids_padded)) == 1)
print()

# Show attention mask of first sequence
# Should be 1s for real tokens then 0s for padding
print("Attention mask (first sequence):")
print(all_attention_masks[0])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# A PyTorch Dataset is a wrapper that lets PyTorch pull
# batches of data during training automatically
# We need to define 3 things:
#   __init__  : store the data
#   __len__   : tell PyTorch how many examples we have
#   __getitem__: tell PyTorch how to get one example by index

class NERDataset(Dataset):

    def __init__(self, input_ids, attention_masks, label_ids):
        # Convert Python lists → PyTorch tensors
        # A tensor is like a numpy array but PyTorch can use it for training
        self.input_ids      = torch.tensor(input_ids,      dtype=torch.long)
        self.attention_masks= torch.tensor(attention_masks,dtype=torch.long)
        self.label_ids      = torch.tensor(label_ids,      dtype=torch.long)

    def __len__(self):
        # How many sequences do we have total?
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Given an index, return that one sequence as a dict
        # PyTorch calls this automatically during training
        return {
            "input_ids":       self.input_ids[idx],
            "attention_mask":  self.attention_masks[idx],
            "labels":          self.label_ids[idx]
        }

# Truncate all sequences to 512 — BioBERT's hard limit
MAX_LEN = 512

input_ids_512  = [ids[:MAX_LEN] for ids in all_input_ids_padded]
masks_512      = [m[:MAX_LEN]   for m   in all_attention_masks]
labels_512     = [l[:MAX_LEN]   for l   in all_label_ids_padded]

# Split into train (80%) and validation (20%)
# We need validation to check if model is learning during training
split       = int(0.8 * len(input_ids_512))  # 800 train, 200 val

train_dataset = NERDataset(
    input_ids_512[:split],
    masks_512[:split],
    labels_512[:split]
)

val_dataset = NERDataset(
    input_ids_512[split:],
    masks_512[split:],
    labels_512[split:]
)

# DataLoader feeds batches to the model during training
# batch_size=8 means model sees 8 sequences at once
# shuffle=True means training order is randomised each epoch
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)

print("Train sequences  :", len(train_dataset))
print("Val sequences    :", len(val_dataset))
print()
print("Train batches    :", len(train_loader))
print("Val batches      :", len(val_loader))
print()

# Check one batch to make sure shapes look right
batch = next(iter(train_loader))
print("One batch — shapes:")
print("  input_ids     :", batch['input_ids'].shape)
print("  attention_mask:", batch['attention_mask'].shape)
print("  labels        :", batch['labels'].shape)

In [ ]:
from transformers import AutoModelForTokenClassification

# Remove local_files_only so it downloads from internet
model = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=5,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded successfully!")
print(f"Total trainable parameters: {total_params:,}")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")
model = model.to(device)
print(f"Model moved to {device}")

In [ ]:
from transformers import AutoModelForTokenClassification

# AutoModelForTokenClassification loads BioBERT
# with a classification head on top
# The classification head is a small layer added after BioBERT
# that takes BioBERT's output for each token and predicts
# one of our 5 labels: O, B-Chemical, I-Chemical, B-Disease, I-Disease

# num_labels=5 tells it how many label types we have
model = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=5,
    local_files_only=True,
    ignore_mismatched_sizes=True
    # ignore_mismatched_sizes=True because BioBERT was originally
    # trained without a classification head — we're adding a new one
    # so sizes won't match perfectly and that's expected
)

# Count total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded successfully!")
print(f"Total trainable parameters: {total_params:,}")
print()

# Check what device we're using (CPU in our case)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

# Move model to device
model = model.to(device)
print(f"Model moved to {device}")

In [ ]:
import pickle
import os

save_dir = r"C:\Users\hp\Desktop\med"
os.makedirs(save_dir, exist_ok=True)

# Save the padded, ready-to-train data
colab_data = {
    "input_ids_512" : input_ids_512,
    "masks_512"     : masks_512,
    "labels_512"    : labels_512
}

save_path = os.path.join(save_dir, "colab_training_data.pkl")

with open(save_path, "wb") as f:
    pickle.dump(colab_data, f)

size_mb = os.path.getsize(save_path) / (1024 * 1024)

print(f"Saved training data to: {save_path}")
print(f"File size: {size_mb:.1f} MB")
print()
print("Contents:")
print(f"  input_ids : {len(input_ids_512)} sequences × 512 tokens")
print(f"  masks     : {len(masks_512)} sequences × 512 tokens")
print(f"  labels    : {len(labels_512)} sequences × 512 tokens")
print()
print("Ready to upload to Google Colab!")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "-q"])

from sentence_transformers import SentenceTransformer
print("sentence-transformers loaded!")

In [ ]:
# Extract all unique MESH ID → concept name pairs from BC5CDR
# We already have this data — it's in the 'normalized' field
# we saw earlier in every entity

mesh_dict = {}  # { "D006973": "hypertensive" }

for split_name in ['train', 'validation', 'test']:
    for row in dataset[split_name]:
        for passage in row['passages']:
            for entity in passage['entities']:
                for norm in entity['normalized']:
                    mesh_id   = norm['db_id']
                    mesh_name = entity['text'][0]
                    # only add if id exists and not seen before
                    if mesh_id and mesh_id not in mesh_dict:
                        mesh_dict[mesh_id] = mesh_name

print(f"Total unique MESH concepts: {len(mesh_dict)}")
print()
print("Sample entries:")
for k, v in list(mesh_dict.items())[:10]:
    print(f"  {k}  →  {v}")